# Auto Encoder

这部分会持续更新。我不会重点写这部分内容，只是经常会遇到诸如 VAE 之类的模型，于是乎就把参考代码实现放这里，方便参考与复用。

## 1. VAE

Variational Autoencoder (VAE) 与 Auto Encoder 的一大不同，就是 encoder 的输出不是一个固定的潜在空间向量，而是一个概率分布的参数。通过随机采样以及 KL 损失约束，模型的潜在空间更具连续性和平滑性。

下面，我将把 World Models(2018) 文章中的 VAE 实现略微改动后放在这里，以供参考。

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [5]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=32): # input: (3, 64, 64)
        super().__init__()
        self.latent_dim = latent_dim
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=0), # -> (32, 31, 31)
            nn.ReLU(),

            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=0), # -> (64, 14, 14)
            nn.ReLU(),

            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=0), # -> (128, 6, 6)
            nn.ReLU(),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=0), # -> (256, 2, 2)
            nn.ReLU(),
        )

        self.fc_mu = nn.Linear(256 * 2 * 2, latent_dim)
        self.fc_logvar = nn.Linear(256 * 2 * 2, latent_dim)

    def forward(self, X):
        # X: (batch_size, 3, 64, 64)
        X = self.conv_layer(X)
        X_flatten = X.flatten(start_dim=1)

        mu = self.fc_mu(X_flatten)
        logvar = self.fc_logvar(X_flatten)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

In [4]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        
        self.fc = nn.Linear(latent_dim, 1024 * 1 * 1) # z -> 1024 * 1 * 1 
        
        self.deconv_layers = nn.Sequential(
            # (1024, 1, 1) -> (128, 5, 5)
            nn.ConvTranspose2d(1024, 128, kernel_size=5, stride=2, padding=0),
            nn.ReLU(),

            # (128, 5, 5) -> (64, 13, 13)
            nn.ConvTranspose2d(128, 64, kernel_size=5, stride=2, padding=0),
            nn.ReLU(),

            # (64, 13, 13) -> (32, 30, 30)
            nn.ConvTranspose2d(64, 32, kernel_size=6, stride=2, padding=0),
            nn.ReLU(),

            # (32, 30, 30) -> (3, 64, 64)
            nn.ConvTranspose2d(32, 3, kernel_size=6, stride=2, padding=0),
            nn.Sigmoid()
        )
    
    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 1024, 1, 1)
        x = self.deconv_layers(x)
        return x

## 2. VQ-VAE

VQ-VAE 在 Genie 中反复用到，最典型的是在 Video Tokenizer 中的使用。

这里我们直接展示在 Genie 中实现的 VQ-VAE。